# Benchmark 06: PyTorch Profiler — Where Does Time Actually Go?

**Goal:** Look inside a single forward pass and see which operations consume time on CPU vs GPU.

**Setup:** Google Colab T4 GPU, HuggingFace Transformers, facebook/opt-125m

**What we're measuring:**
- Which operations (linear layers, attention, etc.) take the most time
- CPU time vs CUDA (GPU) time for each operation
- How many times each operation is called during generation

**Why this matters:**
Every benchmark so far measured *that* something is slow (TTFT, tokens/sec).
The profiler shows *where* the time goes — which operations, how many calls,
CPU overhead vs actual GPU compute. This is the difference between knowing
a car is slow and knowing whether it's the engine, the brakes, or the tires.

## Setup

Run on Colab with T4 GPU runtime (Runtime -> Change runtime type -> T4 GPU)

In [ ]:
!pip install transformers accelerate -q

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.profiler import profile, ProfilerActivity

MODEL = "facebook/opt-125m"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16).to("cuda")

inputs = tokenizer("Explain what a KV cache is", return_tensors="pt").to("cuda")

print("Setup complete")

## Profile a Forward Pass

We generate 20 tokens while the profiler records every operation that runs,
on both CPU and GPU. `key_averages()` groups repeated calls and sums their time.
Sorting by `cuda_time_total` shows which operations consumed the most GPU time.

In [ ]:
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    with torch.no_grad():
        model.generate(**inputs, max_new_tokens=20)

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

## Reading the Output

**Top-level numbers:**
- Self CPU time total: 2.350s
- Self CUDA time total: 58.145ms

The GPU computation itself took 58ms. The whole operation took 2.35 seconds.
The gap, roughly 40x, is overhead, not computation.

**Key rows:**

| Operation | CPU total | CUDA total | # calls | What it is |
|-----------|-----------|------------|---------|------------|
| `aten::linear` | 916ms | 43ms | 1460 | Linear layers (Q/K/V projections, MLP) |
| `aten::addmm` | 793ms | 33ms | 1440 | The matrix multiply underneath `linear` |
| `gemv2T_kernel` | - | 8.9ms | 20 | Actual CUDA kernel doing the math |
| `scaled_dot_product_attention` | 106ms | 4.6ms | 240 | Attention computation |

**The core finding: 1460 calls to `aten::linear`**

Each transformer layer makes roughly 6 linear calls (Q, K, V, output projection,
and 2 in the MLP). With 12 layers and 20 generated tokens:

```
6 calls/layer x 12 layers x 20 tokens ~= 1440 calls
```

This matches the measured 1460. Each call has fixed overhead: Python packages
the instruction, sends it to the GPU driver, waits for the result. With 1460
calls, that overhead dominates total time even though each individual GPU
operation is tiny (43ms / 1460 ~= 0.03ms per call).

## Why This Matters for Production Inference

**CUDA graphs** record the entire sequence of kernel launches once and replay
it as a single instruction. Same GPU work, but eliminates the Python round-trip
for 1459 of the 1460 calls. This is what vLLM's "CUDAGraph capture" step does
at startup (the 39 seconds seen in earlier benchmarks).

**Continuous batching** doesn't reduce the number of calls per forward pass,
but makes each call do work for multiple users at once. The same 1440 calls
now serve N users instead of 1.

Both techniques attack the same root cause visible in this profile: per-call
overhead dominates when the actual GPU computation is small relative to the
number of calls.

## Connection to Previous Benchmarks

- **Roofline model (week 7):** confirmed - most time isn't spent computing,
  it's spent on overhead around small computations
- **vLLM vs Ollama 99x speedup (benchmark 02):** part of that speedup is
  exactly this - CUDA graphs and batching reducing per-call overhead
- **Quantization (benchmark 03):** smaller weights don't reduce call count,
  which is part of why quantization didn't help on this small model - the
  bottleneck wasn't weight size, it was call overhead

## Next: Speculative Decoding
Use a small draft model to predict multiple tokens, verified by the large
model in one pass, directly reducing the number of forward passes (and
therefore the number of these 1460-call cycles) needed per response.